# colab6 — Design B: weights encode Y directly (`relu(w_i)`, no Y in input)

## Background

In colab5 we showed that relabeling the alphabet from `{0, 1}` to `{1, 2}` fixes the dead-gradient problem. But we also noticed that all weights converge to `1` regardless of the target string. This is because the paper's trainable layer computes `relu(w_i * y[i])` — it *multiplies* the weight by the input y-character. Since the correctness condition is `relu(w_i * y[i]) = y[i]`, the weight must always be `1` (identity scaler). The weight doesn't encode Y; the input does.

This is architecturally circular: Y is fed as input, then the network trains to... not modify Y. The weight learns nothing about the target string.

## The fix: Design B

Replace `relu(w_i * y[i])` with `relu(w_i)` — a constant `1` goes into the Dense layer instead of `y[i]`. Now:

- The correctness condition becomes `relu(w_i) = y[i]`, so `w_i = y[i]` (the weight **is** the character value)
- Y is no longer in the model's forward pass — only X is. The network must discover Y purely from edit-distance supervision.
- For `{1, 2}` encoding, all targets are positive, so `relu'(w_i) = 1` everywhere and gradients are always nonzero.
- Expected final weights: `[1, 1]` for DESIRE='11', `[2, 1]` for DESIRE='21', `[2, 1, 2, 1, 1]` for DESIRE='21211'.

The model change is **one line** inside `align_model_for_N`: feed a constant `1` instead of `y_slice` into the trainable Dense layer.

## Setup

In [1]:
import os
!rm -rf /content/thesis-edit-distance-nn

!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git

os.chdir('/content/thesis-edit-distance-nn')

!ls sampledata/

Cloning into 'thesis-edit-distance-nn'...
remote: Enumerating objects: 100, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 100 (delta 56), reused 51 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (100/100), 558.23 KiB | 3.85 MiB/s, done.
Resolving deltas: 100% (56/56), done.
desired_length_2_levenshtein_2.csv
desired_length_2_levenshtein_3.csv
desired_length_2_levenshtein.csv
desired_length_2_levenshtein_relabel_11.csv
desired_length_2_levenshtein_relabel_21.csv
desired_length_5_levenshtein_relabel_21211.csv
predtime_length_10_2_levenshtein.csv
predtime_length_5_4_levenshtein.csv
predtime_length_5_levenshtein.csv


In [2]:
!pip install tensorflow ml_dtypes --upgrade

In [3]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [4]:
import random
import time
import os
import math
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Activation, Input, add, Lambda, Reshape, concatenate, Flatten

## Architecture (Design B — one-line change in `align_model_for_N`)

All frozen modules (`matching_module`, `min_module`, `minimum`) are identical to the original paper.

The only change is in `align_model_for_N`: the `for_gen_dense_*` loop feeds a **constant 1** into the trainable Dense layer instead of `y[i]` from the input. This makes the output `relu(w_i * 1) = relu(w_i)` — the weight itself is the learned y-encoding.

In [5]:
def transform_seqs_to_input(seqA, seqB):
    matching_pairs = []
    input_length_x = 0

    matching_pairs.append([int(seqA[0]), int(seqB[0])])
    if len(seqA) == 1 and len(seqB) == 1:
        return matching_pairs
    else:
        input_length_x = len(seqA)
        match_layers_i = (input_length_x * 2) - 1

    start_i = 1
    end_i = 2

    for l in range(match_layers_i):
        if l < input_length_x - 1:
            i, j = [*reversed(range(0, end_i))], [*range(0, end_i)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    pair = [int(seqA[i[n]]), int(seqB[j[n]])]
                    matching_pairs.append(pair)
            end_i += 1
        else:
            i, j = [*reversed(range(start_i, input_length_x))], [*range(start_i, input_length_x)]
            for n in range(len(i)):
                if j[n] < len(seqB):
                    pair = [int(seqA[i[n]]), int(seqB[j[n]])]
                    matching_pairs.append(pair)
            start_i += 1
            if start_i > len(seqB):
                break

    return matching_pairs


def transform_input_for_generate(input):
    x = []
    y = []
    for pair in input:
        x.append(pair[0])
        y.append(pair[1])
    return [x, y]

In [6]:
def matching_module():
    epsilon = 1

    model = Sequential()
    model.add(Dense(units=2, activation='relu', use_bias=True, input_shape=(2,)))
    model.add(Dense(units=2, activation='relu', use_bias=True))
    model.add(Dense(units=1, activation='relu', use_bias=True))

    w1 = model.layers[0].get_weights()
    w1[0][0][0], w1[0][0][1] = 1.0, -1.0
    w1[0][1][0], w1[0][1][1] = -1.0, 1.0
    w1[1][0], w1[1][1] = 0, 0
    w2 = model.layers[1].get_weights()
    w2[0][0][0], w2[0][0][1] = 1.0, 1.0
    w2[0][1][0], w2[0][1][1] = 1.0, 1.0
    w2[1][0], w2[1][1] = epsilon, -1 * epsilon
    w3 = model.layers[2].get_weights()
    w3[0][0][0], w3[0][1][0] = (1.0/epsilon), -1.0 * (1.0/epsilon)
    w3[1][0] = -1

    model.layers[0].set_weights(w1)
    model.layers[1].set_weights(w2)
    model.layers[2].set_weights(w3)

    model.trainable = False
    return model

In [7]:
def min_module(i, j, k):
    input = Input(shape=(2,))
    x = Dense(2, activation='relu', use_bias=True)(input)
    combined = concatenate([x, input])

    layer_name = 'result_pixel_' + str(i) + str(j) + '_' + str(k)
    z = Dense(1, activation='relu', use_bias=True, name=layer_name)(combined)
    model = Model(inputs=input, outputs=z)

    w1 = model.layers[1].get_weights()
    w1[0][0], w1[0][1] = [-1.0, 1.0], [1.0, -1.0]
    w2 = model.layers[3].get_weights()
    w2[0][0], w2[0][1], w2[0][2], w2[0][3] = -0.5, -0.5, 0.5, 0.5

    model.layers[1].set_weights(w1)
    model.layers[3].set_weights(w2)

    model.trainable = False
    return model


def minimum(i, j):
    input = Input(shape=(3,))
    comp1_pair = Lambda(lambda x: x[:, :2], output_shape=(2,))(input)
    comp2_input = Lambda(lambda x: x[:, 2:], output_shape=(1,))(input)

    m = min_module(i, j, 1)(comp1_pair)
    comp2_pair = concatenate([comp2_input, m])
    output = min_module(i, j, 2)(comp2_pair)

    model = Model(inputs=input, outputs=output)
    model.trainable = False
    return model

In [ ]:
def align_model_for_N(seq_length_x, seq_length_y, matching_pair_number):
    input = Input(shape=(2, matching_pair_number), name='input')

    y = Lambda(lambda t: t[:, 1, :], output_shape=(matching_pair_number,))(input)
    x = Lambda(lambda t: t[:, 0, :], output_shape=(matching_pair_number,))(input)

    out = {}
    start_i = 0
    step = 2
    for i in range(seq_length_y):
        a = start_i
        layername = 'for_gen_dense_' + str(i + 1)
        # === DESIGN B CHANGE ===
        # Feed a constant 1 instead of y[i]. The Dense output becomes relu(w_i * 1) = relu(w_i).
        # The weight itself IS the y-encoding — no y in the forward pass.
        ones_input = Lambda(lambda t, a=a: tf.ones_like(t[:, a:a+1]), output_shape=(1,))(y)
        z = Dense(1, activation='relu', name=layername, use_bias=False)(ones_input)
        out[layername] = z
        start_i += step
        step += 1

    pair_i = 1
    calc_layer = (seq_length_x * 2) - 1
    test_dict = {}

    y_dense_layer_name = 'for_gen_dense_1'
    densed_y = out[y_dense_layer_name]
    x_char = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(x)

    debug_name = 'matching_debug_1'
    pair_11 = concatenate([x_char, densed_y], name=debug_name)

    ext_gaps = Dense(2, activation='relu', name='first_calc_gap_layer')(pair_11)

    min1 = min_module(1, 1, 1)(ext_gaps)
    matching1 = matching_module()(pair_11)
    combined = concatenate([min1, matching1])
    z = min_module(1, 1, 2)(combined)
    result_pixel_11 = concatenate([ext_gaps, z], name='input_pixel_1_1')

    pair_i = 2

    if seq_length_x == 1 and seq_length_y == 1:
        output = z
        return Model(inputs=input, outputs=output)
    else:
        test_dict['input_pixel_1_1'] = result_pixel_11
        test_dict['result_pixel_1_1'] = z

        comp_i_val, comp_j_val = 1, 2
        start_sentinel, end_sentinel = 1, 2
        unbalance_flag = True

        for calc_layer_i in range(calc_layer):
            if calc_layer_i < seq_length_x - 1:
                comp_i_val, comp_j_val = start_sentinel, end_sentinel
                while comp_i_val <= end_sentinel:
                    if comp_i_val <= seq_length_y:
                        input_layer_name = 'input_' + str(comp_i_val) + '_' + str(comp_j_val)
                        before_input_layer_name = 'before_input_' + str(comp_i_val) + '_' + str(comp_j_val)

                        c = pair_i
                        y_i = comp_i_val
                        y_dense_layer_name = 'for_gen_dense_' + str(y_i)
                        densed_y = out[y_dense_layer_name]

                        x_char = Lambda(lambda t, c=c: t[:, c-1:c], output_shape=(1,))(x)

                        debug_name = 'matching_debug_' + str(c)
                        pair = concatenate([x_char, densed_y], name=debug_name)
                        matching = matching_module()(pair)

                        if comp_i_val == 1:
                            previous_input_pixel_name = 'input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)
                            previous_result_pixel_name = 'result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)
                            previous_input = test_dict[previous_input_pixel_name]
                            previous_result = test_dict[previous_result_pixel_name]
                            g = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(previous_input)
                            before_input = concatenate([g, previous_result, matching], name=before_input_layer_name)

                        elif comp_j_val == 1:
                            previous_input_pixel_name = 'input_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)
                            previous_result_pixel_name = 'result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)
                            previous_input = test_dict[previous_input_pixel_name]
                            previous_result = test_dict[previous_result_pixel_name]
                            g = Lambda(lambda t: t[:, 0:1], output_shape=(1,))(previous_input)
                            before_input = concatenate([g, previous_result, matching], name=before_input_layer_name)

                        else:
                            previous_result1 = test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)]
                            previous_result2 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)]
                            previous_result3 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val - 1)]
                            before_input = concatenate([previous_result1, previous_result2, previous_result3, matching], name=before_input_layer_name)

                        input_pixel = Dense(3, activation='relu', name=input_layer_name)(before_input)
                        result_pixel = minimum(comp_i_val, comp_j_val)(input_pixel)

                        test_dict['input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = input_pixel
                        test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = result_pixel

                        if unbalance_flag:
                            unbalance_flag = False

                    comp_i_val += 1
                    comp_j_val -= 1
                    pair_i += 1
                    if unbalance_flag:
                        pair_i -= 1
                    unbalance_flag = True

                if end_sentinel + 1 <= seq_length_x:
                    end_sentinel += 1

            else:
                start_sentinel += 1
                comp_i_val, comp_j_val = start_sentinel, end_sentinel

                while comp_i_val <= end_sentinel:
                    if comp_i_val <= seq_length_y:
                        before_input_layer_name = 'before_input_' + str(comp_i_val) + '_' + str(comp_j_val)
                        input_layer_name = 'input_' + str(comp_i_val) + '_' + str(comp_j_val)

                        c = pair_i
                        y_dense_layer_name = 'for_gen_dense_' + str(comp_i_val)
                        densed_y = out[y_dense_layer_name]

                        x_char = Lambda(lambda t, c=c: t[:, c-1:c], output_shape=(1,))(x)
                        debug_name = 'matching_debug_' + str(c)
                        pair = concatenate([x_char, densed_y], name=debug_name)
                        matching = matching_module()(pair)

                        previous_result1 = test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val - 1)]
                        previous_result2 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val)]
                        previous_result3 = test_dict['result_pixel_' + str(comp_i_val - 1) + '_' + str(comp_j_val - 1)]
                        before_input = concatenate([previous_result1, previous_result2, previous_result3, matching], name=before_input_layer_name)

                        input_pixel = Dense(3, activation='relu', name=input_layer_name)(before_input)
                        result_pixel = minimum(comp_i_val, comp_j_val)(input_pixel)

                        test_dict['input_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = input_pixel
                        test_dict['result_pixel_' + str(comp_i_val) + '_' + str(comp_j_val)] = result_pixel

                        if unbalance_flag:
                            unbalance_flag = False

                    comp_i_val += 1
                    comp_j_val -= 1
                    pair_i += 1
                    if unbalance_flag:
                        pair_i -= 1
                    unbalance_flag = True

                    if start_sentinel == end_sentinel:
                        return Model(inputs=input, outputs=result_pixel)

In [9]:
def set_weight_for_debug(model, seq_len_x, seq_len_y, matching_pair):
    """Set analytical weights on frozen layers. Trainable for_gen_dense weights remain at random init."""
    print('setting frozen weights ...')

    w = model.get_layer('first_calc_gap_layer').get_weights()
    w[0][0][0], w[0][0][1] = 0, 0
    w[0][1][0], w[0][1][1] = 0, 0
    w[1][0], w[1][1] = 2, 2
    model.get_layer('first_calc_gap_layer').set_weights(w)
    model.get_layer('first_calc_gap_layer').trainable = False

    if seq_len_x > 1:
        calc_layer = (seq_len_x * 2) - 1
        comp_i, comp_j = 1, 2
        start_sentinel, end_sentinel = 1, 2

        for calc_layer_i in range(calc_layer):
            if calc_layer_i < seq_len_x - 1:
                comp_i, comp_j = start_sentinel, end_sentinel
                while comp_i <= end_sentinel:
                    if comp_i <= seq_len_y:
                        input_layer_name = 'input_' + str(comp_i) + '_' + str(comp_j)
                        w = model.get_layer(input_layer_name).get_weights()
                        if comp_i == 1:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 1
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, -1
                        elif comp_j == 1:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 1
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, -1
                        else:
                            w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 0
                            w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                            w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                            w[0][3][0], w[0][3][1], w[0][3][2] = 0, 0, 1
                            w[1][0], w[1][1], w[1][2] = 1, 1, 0

                        model.get_layer(input_layer_name).set_weights(w)
                        model.get_layer(input_layer_name).trainable = False

                    comp_i, comp_j = (comp_i + 1), (comp_j - 1)
                if end_sentinel + 1 <= seq_len_x:
                    end_sentinel += 1
            else:
                start_sentinel = start_sentinel + 1
                comp_i, comp_j = start_sentinel, end_sentinel

                while comp_i <= end_sentinel:
                    if comp_i <= seq_len_y:
                        input_layer_name = 'input_' + str(comp_i) + '_' + str(comp_j)
                        w = model.get_layer(input_layer_name).get_weights()
                        w[0][0][0], w[0][0][1], w[0][0][2] = 1, 0, 0
                        w[0][1][0], w[0][1][1], w[0][1][2] = 0, 1, 0
                        w[0][2][0], w[0][2][1], w[0][2][2] = 0, 0, 1
                        w[0][3][0], w[0][3][1], w[0][3][2] = 0, 0, 1
                        w[1][0], w[1][1], w[1][2] = 1, 1, 0
                        model.get_layer(input_layer_name).set_weights(w)
                        model.get_layer(input_layer_name).trainable = False

                    comp_i, comp_j = (comp_i + 1), (comp_j - 1)


def froozen_align_model(model):
    print('freezing all non-trainable layers ...')
    for layer in model.layers:
        if 'for_gen_dense' in layer.name:
            layer.trainable = True
        else:
            layer.trainable = False

## SGD training

Same vanilla SGD loop as colab5. The only difference is what we expect the weights to converge to:

| Target (relabeled) | Expected final weights |
|---|---|
| `'11'` | `[1, 1]` (both chars are 1) |
| `'21'` | `[2, 1]` (weights ARE the characters now) |
| `'21211'` | `[2, 1, 2, 1, 1]` |

Also note: the gradient `d(relu(w_i))/dw_i = 1` for all positive `w_i`, regardless of `y[i]`. So unlike colab5, convergence rate should be **uniform** across positions — no more asymmetry between `y=1` and `y=2` dimensions. The only variation in speed should come from initialization distance to target.

In [ ]:
def training(desire, csv_path, epochs=20, learning_rate=0.1, seed=None):
    """
    Vanilla SGD training of the for_gen_dense weights.
    In Design B, weights should converge to the actual character values of `desire`.
    """
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        tf.random.set_seed(seed)

    LEN = len(desire)

    lines = []
    with open(csv_path, 'r') as f:
        for line in f:
            lines.append(line.rstrip('\n'))

    print('=== training data ===')
    for line in lines:
        print(' ', line)

    sp = lines[0].split(',')
    x, y = sp[0], sp[1]
    pairs = transform_seqs_to_input(x, y)
    SEQ_LEN_X = len(x)
    SEQ_LEN_Y = len(y)
    PAIRS_LEN = len(pairs)

    model = align_model_for_N(SEQ_LEN_X, SEQ_LEN_Y, PAIRS_LEN)
    set_weight_for_debug(model, SEQ_LEN_X, SEQ_LEN_Y, PAIRS_LEN)
    froozen_align_model(model)

    init_trained_weights = []
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        w = model.get_layer(lname).get_weights()
        w[0][0][0] = random.uniform(0.1, 2.5)
        model.get_layer(lname).set_weights(w)
        init_trained_weights.append(float(w[0][0][0]))

    target_weights = [int(c) for c in desire]
    print('init   weights:', init_trained_weights)
    print('target weights:', target_weights)

    progress_weights = [list(init_trained_weights)]
    progress_losses = []

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)
    loss_fn = tf.keras.losses.MeanSquaredError()

    for epoch in range(epochs):
        loss = tf.Variable(0.0, name='loss')
        with tf.GradientTape() as tape:
            for line in lines:
                sp = line.split(',')
                xs, ys, true_score = sp[0], sp[1], int(sp[2])
                inp = transform_seqs_to_input(xs, ys)
                inp = transform_input_for_generate(inp)
                inp = tf.constant([inp], dtype=tf.float32)
                logit = model(inp, training=True)
                loss = loss + loss_fn(true_score, logit)
            batch_loss = loss / len(lines)
            grads = tape.gradient(batch_loss, model.trainable_weights)
            optimizer.apply_gradients(zip(grads, model.trainable_weights))

        progress_losses.append(float(batch_loss.numpy()))
        cur = []
        for i in range(LEN):
            lname = 'for_gen_dense_' + str(i + 1)
            cur.append(float(model.get_layer(lname).get_weights()[0][0][0]))
        progress_weights.append(cur)
        print(f'epoch {epoch:3d} | batch_loss = {float(batch_loss.numpy()):.6f} | weights = {[round(w, 4) for w in cur]}')

    final_weights = []
    for i in range(LEN):
        lname = 'for_gen_dense_' + str(i + 1)
        final_weights.append(float(model.get_layer(lname).get_weights()[0][0][0]))

    print('init   weights:', init_trained_weights)
    print('final  weights:', final_weights)
    print('target weights:', target_weights)

    return {
        'desire': desire,
        'target_weights': target_weights,
        'init_weights': init_trained_weights,
        'final_weights': final_weights,
        'progress_weights': progress_weights,
        'progress_losses': progress_losses,
        'model': model,
    }


def plot_training(result, title=''):
    progress_weights = np.array(result['progress_weights'])
    losses = result['progress_losses']
    target = result['target_weights']
    LEN = progress_weights.shape[1]
    epochs = progress_weights.shape[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    for w_i in range(LEN):
        axes[0].plot(range(epochs), progress_weights[:, w_i], marker='o', markersize=3,
                     label=f'w_{w_i+1} (target={target[w_i]})', color=color_cycle[w_i])
        axes[0].axhline(target[w_i], color=color_cycle[w_i], linestyle='dotted', alpha=0.5)
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('weight value')
    axes[0].set_title('weight trajectories')
    axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

    axes[1].plot(range(1, len(losses) + 1), losses, marker='o', markersize=3, color='red')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('MSE loss')
    axes[1].set_title('training loss')
    axes[1].set_ylim(bottom=0)

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Experiment 1 — DESIRE='11'

Baseline. Both characters are `1`, so target weights are `[1, 1]` — same as colab5. This verifies Design B doesn't break anything.

In [ ]:
result_11 = training(
    desire='11',
    csv_path='/content/thesis-edit-distance-nn/sampledata/desired_length_2_levenshtein_relabel_11.csv',
    epochs=20,
    learning_rate=0.1,
    seed=42,
)
plot_training(result_11, title="DESIRE='11' | Design B")

## Experiment 2 — DESIRE='21' (the headline test)

Target weights are `[2, 1]` — the weights should now reflect the actual character values of the target string. This is the first time we expect a weight to converge to something other than 1.

In [ ]:
result_21 = training(
    desire='21',
    csv_path='/content/thesis-edit-distance-nn/sampledata/desired_length_2_levenshtein_relabel_21.csv',
    epochs=20,
    learning_rate=0.1,
    seed=42,
)
plot_training(result_21, title="DESIRE='21' (from '01') | Design B")

## Experiment 3 — DESIRE='21211' (5 characters)

Target weights are `[2, 1, 2, 1, 1]`. The network must discover the full target string from 32 training examples (all length-5 binary strings in `{1, 2}` and their edit distances to `'21211'`).

Since `d(relu(w_i))/dw_i = 1` for all positive `w_i`, the gradient magnitude is the same across all positions. Convergence speed should depend only on initialization distance to target.

In [ ]:
result_21211 = training(
    desire='21211',
    csv_path='/content/thesis-edit-distance-nn/sampledata/desired_length_5_levenshtein_relabel_21211.csv',
    epochs=20,
    learning_rate=0.1,
    seed=42,
)
plot_training(result_21211, title="DESIRE='21211' (from '01011') | Design B")

## What to look for

1. **Weights should converge to the actual character values** — `[1,1]` for exp 1, `[2,1]` for exp 2, `[2,1,2,1,1]` for exp 3. This is the key difference from colab5 where everything converged to 1.
2. **Loss should drop to ~0** within a small number of epochs in all three cases.
3. **No weight should be stuck** — all dimensions have gradient factor 1 (from `relu'(w_i) = 1`), so convergence should be uniform.
4. **You can read off the target string from the trained weights.** The weights now carry interpretable meaning: `w_i ≈ 2` means position `i` is character `2` (original `0`), `w_i ≈ 1` means character `1`. The network has genuinely *learned* what Y is from edit-distance supervision alone.